In [ ]:
import numpy as np
import plotly.graph_objects as go

# Echo times
TE = np.arange(0, 302, 2)

# Fixed T2 values
T2_MW = 20
T2_IEW = 80

# MWF values
mwf_values = np.arange(1, 51, 1)

# Default values
initial_healthy = 20
initial_demyelinated = 5

fig = go.Figure()

for healthy_mwf in mwf_values:
    for demyelinated_mwf in mwf_values:

        # Healthy tissue
        A_mw = healthy_mwf / 100
        A_iew = 1 - A_mw

        healthy_signal = (
            A_mw * np.exp(-TE / T2_MW)
            + A_iew * np.exp(-TE / T2_IEW)
        )

        # Demyelinated tissue
        A_mw_d = demyelinated_mwf / 100
        A_iew_d = 1 - A_mw_d

        demyelinated_signal = (
            A_mw_d * np.exp(-TE / T2_MW)
            + A_iew_d * np.exp(-TE / T2_IEW)
        )

        visible = (
            healthy_mwf == initial_healthy
            and demyelinated_mwf == initial_demyelinated
        )

        fig.add_trace(
            go.Scatter(
                x=TE,
                y=healthy_signal,
                name="Healthy tissue",
                visible=visible
            )
        )

        fig.add_trace(
            go.Scatter(
                x=TE,
                y=demyelinated_signal,
                name="Demyelinated tissue",
                visible=visible
            )
        )

steps = []

for i, healthy_mwf in enumerate(mwf_values):
    for j, demyelinated_mwf in enumerate(mwf_values):

        index = i * len(mwf_values) + j

        visibility = [False] * len(fig.data)
        visibility[2 * index] = True
        visibility[2 * index + 1] = True

        steps.append(
            dict(
                method="update",
                args=[
                    {"visible": visibility},
                    {
                        "title": (
                            f"Healthy MWF = {healthy_mwf}%, "
                            f"Demyelinated MWF = {demyelinated_mwf}%"
                        )
                    }
                ],
                label=f"{healthy_mwf}/{demyelinated_mwf}"
            )
        )

fig.update_layout(
    title="Healthy vs Demyelinated Tissue",
    xaxis_title="Echo time (ms)",
    yaxis_title="Normalized signal",
    height=500,
    sliders=[
        dict(
            active=0,
            currentvalue={"prefix": "Healthy MWF / Demyelinated MWF: "},
            steps=steps
        )
    ]
)

fig.show()